# Group 2 Project
## Team Members: Cedric Tong, Daniel Graham, Justin Gourneau, Tyler Barton, Yashin Rodriguez

### Narrative
We are leveraging the Kaggle dataset located at 
https://www.kaggle.com/datasets/blastchar/telco-customer-churn which is a synthetic dataset comprised of records from a telecommunications company with a variety of data features, the most notable of which is which customers churned. Our aim is to use this dataset and analyze which features are good predictors for customer churn.

# Imports

In [41]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as stats
from sklearn.preprocessing import MinMaxScaler

# Load the Dataset

In [66]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


# Perform Preprocessing

In [67]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

df['TotalCharges'] = df['TotalCharges'].replace(' ', 0)
df['MonthlyCharges'] = df['MonthlyCharges'].replace(' ', 0)
df['tenure'] = df['tenure'].replace(' ', 0)
df['TotalCharges'] = df['TotalCharges'].astype(float)
df['MonthlyCharges'] = df['MonthlyCharges'].astype(float)
df['tenure'] = df['tenure'].astype(float)
scaler = MinMaxScaler()
df['tenure'] = scaler.fit_transform(df[['tenure']])
df['MonthlyCharges'] = scaler.fit_transform(df[['MonthlyCharges']])
df['TotalCharges'] = scaler.fit_transform(df[['TotalCharges']])
categorical_features = ["Contract", "PaymentMethod", "OnlineSecurity", "TechSupport"]

# Split the Data For Training Prep

In [68]:
from sklearn.model_selection import train_test_split
X = df[categorical_features]
y_str = df["Churn"]
y = y_str.map({"No": 0, "Yes": 1}).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=11, 
    stratify=y
)

# Train the Model

In [69]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

pos_weight = 2

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ],
    remainder="drop",
)

model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        (
            "classifier",
            GradientBoostingClassifier(
                n_estimators=500,
                learning_rate=0.05,
                max_depth=3,
                subsample=0.8,
                random_state=11,
            ),
        ),
    ]
)

clf = model.fit(
    X_train,
    y_train,
    classifier__sample_weight=np.where(y_train == 1, pos_weight, 1),
)
y_proba = clf.predict_proba(X_test)[:, 1]

# Print Results

In [70]:
from sklearn.metrics import roc_auc_score, f1_score, precision_score, confusion_matrix, classification_report, accuracy_score, recall_score
threshold = 0.4
y_pred = (y_proba >= threshold).astype(int)

print("Model: Gradient Boost Classifier")
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")
print(f"F1: {f1_score(y_test, y_pred)}")
print(f"Precision: {precision_score(y_test, y_pred)}")
print(f"Recall Score: {recall_score(y_test, y_pred)}")
print(f"ROC AUC: {roc_auc_score(y_test, y_pred)}")

cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=["True 0 (No)", "True 1 (Yes)"],
    columns=["Pred 0 (No)", "Pred 1 (Yes)"],
)
print("Confusion Matrix:\n", cm_df)

# print(f"Confusion Matrix:\n {confusion_matrix(y_test, y_pred)}")
print(f"Classification Report:\n {classification_report(y_test, y_pred)}")

Model: Gradient Boost Classifier
Accuracy: 0.7267565649396736
F1: 0.6059365404298874
Precision: 0.49087893864013266
Recall Score: 0.7914438502673797
ROC AUC: 0.7474127463897287
Confusion Matrix:
               Pred 0 (No)  Pred 1 (Yes)
True 0 (No)           728           307
True 1 (Yes)           78           296
Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.70      0.79      1035
           1       0.49      0.79      0.61       374

    accuracy                           0.73      1409
   macro avg       0.70      0.75      0.70      1409
weighted avg       0.79      0.73      0.74      1409



# Save Model For Deployment

In [71]:
import joblib
from pathlib import Path

model_path = Path('deployedmodel/churn_model.pkl')
model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(clf, model_path)
print(f"Saved model to {model_path.resolve()}")

Saved model to /Users/danielgraham/dev/CSUN/COMP541/deployedmodel/churn_model.pkl
